# 01. 알약 데이터 EDA (통합본)

팀원별 EDA 노트북 6개를 주제별로 합친 노트북입니다. 같은 주제는 한 셀을 기준본으로 두고
나머지 노트북의 장점을 기준본에 녹였습니다. 섹션마다 첫 셀에 원작자와 원본 노트북을 적었습니다.

## 실행 방법

- `uv sync` 후 VS Code나 `uv run jupyter lab`에서 프로젝트 `.venv` 커널로 엽니다.
- 위에서 아래로 한 번 실행(Restart & Run All)합니다. 작업 폴더는 프로젝트 루트와 `notebooks/` 모두 됩니다.
- 원본은 `data/raw/`에서 읽기만 합니다. `ann_df`·`img_df`·`cat_df`는 원본 상태로 유지하고,
  보정 비교는 별도 `ann_corrected_df`에서 합니다.
- 커밋 전에 출력을 지웁니다(Clear All Outputs).

## 실행 옵션 (1장 설정 셀)

| 옵션 | 기본값 | 설명 |
|---|---|---|
| `RUN_HEAVY` | `False` | 전체 이미지 SHA256·디코딩, 전체 박스 GrabCut 같은 무거운 연산을 실행합니다. `False`이면 입력·버전·파라미터가 일치하는 캐시만 재사용하고, 없으면 `미실행`으로 표시합니다. `True`이면 캐시와 관계없이 다시 계산합니다. |
| `SAVE_OUTPUTS` | `False` | 그림·보고서·캐시 파일을 저장합니다. `False`이면 파일을 쓰지 않습니다. 켜도 원본은 수정하지 않습니다. |
| `SEED` | `42` | 무작위 연산 전체에 쓰는 seed입니다. |

`미실행` 결과는 0건이나 검사 통과로 해석하지 않습니다. 후속 셀은 결과 상태를 확인한 뒤 집계합니다.

## 데이터 상태 표시

표와 그림 제목 앞에 데이터 상태를 붙입니다. `[원본]`은 `data/raw/`를 그대로 읽은 결과,
`[보정]`은 보정 기록을 메모리에서 적용한 결과입니다.

## 섹션별 실행 시간

| 섹션 | `RUN_HEAVY=False` | `RUN_HEAVY=True` |
|---|---|---|
| 1–2. 설정·데이터 로드 | 약 3초 (Apple Silicon Mac) | 같음 |
| 3–9 | 이식 후 측정 | 이식 후 측정 |

## 목차

0. 개요
1. 환경 설정
2. 데이터 로드
3. 데이터 규모와 구조
4. 결측
5. 라벨 오류와 이상치
6. 분포
7. 촬영 조건
8. 데이터 누수와 분할
9. 결론과 다음 작업

## 원본 노트북

| 노트북 | 작성자 |
|---|---|
| `EDA_jang.ipynb` | younghoon1215 |
| `JJAM99_eda.ipynb` | potato |
| `dongwook_eda.ipynb` | def-uk |
| `eda_kwon.ipynb` | gwon753-dev |
| `eda_sohee.ipynb`, `sohee_eda.ipynb` | bravedolphinhere |

## 1. 환경 설정

> 출처: dongwook_eda.ipynb 셀 1 (def-uk), eda_kwon.ipynb 셀 2, 35 (gwon753-dev),
> sohee_eda.ipynb 셀 1, 3 (bravedolphinhere)

- 경로는 모두 `src/config.py`의 `PROJECT_ROOT` 기준입니다.
- 한글 폰트는 Malgun Gothic → AppleGothic → NanumGothic → Noto Sans CJK KR 순서로 찾습니다.
  WSL에서는 Windows의 맑은 고딕 파일을 등록합니다.
- 무작위 연산은 모두 `SEED`를 씁니다(`random_state=SEED`, `np.random.default_rng(SEED)`).

In [ ]:
import sys
from pathlib import Path

# 노트북은 notebooks/에서 실행되므로 프로젝트 루트를 import 경로에 추가한다.
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

from data_process.data_processing import (
    build_cat_df,
    build_img_df,
    list_images,
    load_annotations,
    load_schema,
)
from src.config import PROJECT_ROOT
from visualization import set_korean_font

SEED = 42
RUN_HEAVY = False  # True면 무거운 연산을 캐시와 관계없이 다시 계산한다.
SAVE_OUTPUTS = False  # True일 때만 그림·보고서·캐시 파일을 쓴다.
NOT_RUN = "미실행"  # 생략한 분석의 결과 상태. 0건이나 통과로 해석하지 않는다.

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"  # 받은 원본 그대로 (읽기만 함)
# 압축을 푼 위치에 따라 sprint_ai_project1_data 폴더가 한 단계 더 있을 수 있다.
if (RAW_DIR / "sprint_ai_project1_data").is_dir():
    RAW_DIR = RAW_DIR / "sprint_ai_project1_data"
PROCESSED_DIR = DATA_DIR / "processed"  # 보정본·보정 기록
CACHE_DIR = PROCESSED_DIR / "eda_cache"  # 무거운 연산 캐시. SAVE_OUTPUTS=True일 때만 쓴다.

TRAIN_IMG_DIR = RAW_DIR / "train_images"
TRAIN_ANN_DIR = RAW_DIR / "train_annotations"
TEST_IMG_DIR = RAW_DIR / "test_images"
for folder in [TRAIN_IMG_DIR, TRAIN_ANN_DIR, TEST_IMG_DIR]:
    if not folder.is_dir():
        raise FileNotFoundError(f"데이터 폴더가 없습니다: {folder.relative_to(PROJECT_ROOT)}")

np.random.seed(SEED)
pd.set_option("display.max_columns", 60)
font = set_korean_font()
if font is None:
    print("한글 폰트를 찾지 못했습니다. 약 이름이 네모로 보일 수 있습니다.")

print(f"SEED={SEED}, RUN_HEAVY={RUN_HEAVY}, SAVE_OUTPUTS={SAVE_OUTPUTS}, 폰트={font}")
print(f"원본 데이터: {RAW_DIR.relative_to(PROJECT_ROOT).as_posix()}/")

## 2. 데이터 로드

> 출처: dongwook_eda.ipynb 셀 0–3 (def-uk) 기준. eda_kwon.ipynb 셀 1–2 (gwon753-dev)의
> 디스크 기준 `img_df`·`cat_df`·`test_files`, sohee_eda.ipynb 셀 2, 7 (bravedolphinhere)의
> JSON 내부 참조 연결과 읽기 오류 기록을 합쳤습니다. 로더는 `data_process/data_processing.py`에 있습니다.

- annotation JSON 하나 = 이미지 1장 × 알약 1개입니다. 같은 이미지의 알약이 약 코드별 폴더에 흩어져 있습니다.
  경로: `train_annotations/<조합>_json/<약코드>/<이미지이름>.json`
- bbox는 COCO 형식 `[x, y, w, h]`(왼쪽 위 기준, 픽셀)입니다.
- 이미지 메타데이터는 JSON 안의 `annotation.image_id`로 연결합니다. 전역 `image_id`로 합치지 않습니다.
- 라벨 식별자는 `(source_json, annotation_id)`입니다. `source_json`은 `train_annotations/` 기준 상대 경로입니다.
- bbox 값이 비어 있는 라벨도 빼지 않고 `x, y, w, h`를 NaN으로 둡니다(4.1에서 확인).
- 읽기·연결 실패는 `load_error_df`에 남기고, 분석에서 제외된 입력 수를 보여 줍니다.

| 표 | 한 행 | 설명 |
|---|---|---|
| `ann_df` | 원본 bbox 1개 | 이미지 메타데이터 + 약 정보 + bbox. 원본 상태를 유지합니다. |
| `img_df` | 디스크의 train 이미지 1장 | 라벨 없는 이미지도 포함합니다(`n_boxes=0`). `combo`는 파일 이름의 알약 조합입니다. |
| `cat_df` | 클래스 1개 | `category_id` ↔ 약 이름, bbox 수, 이미지 수 |
| `test_files` | test 이미지 1장 | 파일 이름 목록 |
| `load_error_df` | 오류 1건 | `stage`: `parse`(JSON 읽기), `image_link`, `category_link`, `empty`(annotation 없음) |

In [ ]:
ann_df, load_error_df = load_annotations(TRAIN_ANN_DIR)
img_df = build_img_df(ann_df, list_images(TRAIN_IMG_DIR))
cat_df = build_cat_df(ann_df)
test_files = [p.name for p in list_images(TEST_IMG_DIR)]

# 제외된 입력: parse 오류는 JSON 파일 전체, image_link 오류는 annotation 하나가 빠진다.
n_json = sum(1 for _ in TRAIN_ANN_DIR.rglob("*.json"))
excluded = load_error_df[load_error_df["excluded"]]
n_json_excluded = (excluded["stage"] == "parse").sum()
n_ann_excluded = (excluded["stage"] == "image_link").sum()
n_no_image = (~ann_df["file_name"].isin(img_df["file_name"])).sum()

print(f"[원본] annotation JSON: {n_json}개 (읽기 실패로 제외: {n_json_excluded}개)")
print(
    f"[원본] bbox: {len(ann_df)}개 (연결 실패로 제외: {n_ann_excluded}개), 클래스: {len(cat_df)}개"
)
print(
    f"[원본] train 이미지: {len(img_df)}장 (bbox 없음: {(img_df['n_boxes'] == 0).sum()}장), "
    f"test 이미지: {len(test_files)}장"
)
print(f"[원본] 디스크에 이미지 파일이 없는 bbox: {n_no_image}개")
print(f"로드 오류: {len(load_error_df)}건")
if not load_error_df.empty:
    display(load_error_df)

In [ ]:
display(ann_df.head(3))
display(img_df.head(3))
display(cat_df.head(3))

### 2.1 데이터 정의서

> 출처: eda_kwon.ipynb 셀 3 (gwon753-dev) 기준. dongwook_eda.ipynb 셀 9 (def-uk)의
> 스키마 필수여부 읽기를 합쳤습니다.

먼저 `data/README.md`의 스키마 표와 실제 JSON 키를 대조하고, `ann_df` 컬럼마다 필수여부(M 필수,
O 선택, C 조건부)와 스키마 설명을 붙입니다. 결측 비율은 4.3에서 봅니다.

In [ ]:
schema_df = load_schema(DATA_DIR / "README.md")
# 최상위 키(images, annotations 등)는 field가 비어 있어 section만 키로 쓴다.
schema_df["key"] = (schema_df["section"] + "." + schema_df["field"]).str.rstrip(".")

# ann_df 컬럼 → 스키마 키. 로더가 바꾸거나 나눈 컬럼만 적고, 나머지는 images[]의 같은 이름.
LOADER_COLS = {
    "image_id": "images.id",
    "source_json": None,  # 로더가 붙인 원본 JSON 상대 경로
    "annotation_id": "annotations.id",
    "category_id": "annotations.category_id",
    "category_name": "categories.name",
    "x": "annotations.bbox",
    "y": "annotations.bbox",
    "w": "annotations.bbox",
    "h": "annotations.bbox",
    "area": "annotations.area",
    "iscrowd": "annotations.iscrowd",
    "ignore": "annotations.ignore",
    "has_segmentation": "annotations.segmentation",
}
col_to_key = {c: LOADER_COLS.get(c, f"images.{c}") for c in ann_df.columns}

json_image_keys = {k for c, k in col_to_key.items() if k and k.startswith("images.")}
is_image_field = (schema_df["section"] == "images") & (schema_df["field"] != "")
schema_image_keys = set(schema_df.loc[is_image_field, "key"])
print("스키마에 없는 JSON 키:", sorted(json_image_keys - schema_image_keys) or "없음")
print("JSON에 없는 스키마 키:", sorted(schema_image_keys - json_image_keys) or "없음")

definition_df = pd.DataFrame(
    {
        "컬럼": ann_df.columns,
        "스키마 키": [col_to_key[c] for c in ann_df.columns],
        "dtype": ann_df.dtypes.astype(str).to_numpy(),
        "고유값 수": ann_df.nunique().to_numpy(),
        "예시": [ann_df[c].dropna().iloc[0] if ann_df[c].notna().any() else None for c in ann_df],
    }
)
definition_df = definition_df.merge(
    schema_df[["key", "type", "required", "description"]],
    left_on="스키마 키",
    right_on="key",
    how="left",
).drop(columns="key")
definition_df = definition_df.rename(
    columns={"type": "스키마 타입", "required": "필수여부", "description": "스키마 설명"}
)
with pd.option_context("display.max_rows", 100, "display.max_colwidth", 40):
    display(definition_df)

아래는 kwon이 값을 보고 정리한 컬럼 해석입니다. 수치는 원본 상태 관측값이며 3–4장에서 다시 확인합니다.
필수여부는 위 표를 따릅니다.

### 이미지 정보

| 컬럼명 | 데이터 타입 | 설명 |
|---|---|---|
| file_name | str | 이미지 파일명. 이미지 단위의 키로 사용됩니다. 앞부분(`K-003351-018357-036637`)에 사진에 찍힌 약 코드가 들어 있습니다. |
| imgfile | str | 이미지 파일명. `file_name`과 같은 값입니다. 스키마 표에는 없는 키입니다. |
| width | int | 이미지 너비(px). 전부 976입니다. |
| height | int | 이미지 높이(px). 전부 1280입니다. |
| image_id | int | JSON 안의 이미지 아이디(`images[].id`)입니다. |
| source_json | str | 이 행을 읽어 온 annotation JSON의 상대 경로입니다(`train_annotations/` 기준). |

### 촬영 조건

| 컬럼명 | 데이터 타입 | 설명 |
|---|---|---|
| back_color | str | 배경색. 전부 `연회색 배경`입니다. |
| light_color | str | 조명 색. 전부 `주백색`입니다. |
| camera_la | int | 카메라 촬영 각도(위도). 70, 75, 90 세 가지입니다. |
| camera_lo | int | 카메라 회전 각도(경도). 전부 0입니다. |
| size | int | 촬영 크기 설정값. 전부 200입니다. |
| drug_dir | str | 촬영된 알약 면. `앞면` 또는 `뒷면`입니다. |
| drug_S | str | 알약 상태. 전부 `정상알약`입니다. |

### 정답 (타깃)

| 컬럼명 | 데이터 타입 | 설명 |
|---|---|---|
| annotation_id | int | 어노테이션 아이디. `source_json`과 함께 라벨 하나를 가리킵니다. |
| category_id | int | **타깃 변수(클래스)**. 약 종류 코드이며 56종입니다. `drug_N`의 숫자 부분과 같습니다. |
| category_name | str | JSON `categories[].name`의 약 이름입니다. |
| x | int | **타깃 변수(bbox)**. 박스 왼쪽 위 꼭짓점의 x 좌표(px)입니다. |
| y | int | **타깃 변수(bbox)**. 박스 왼쪽 위 꼭짓점의 y 좌표(px)입니다. |
| w | int | **타깃 변수(bbox)**. 박스 너비(px)입니다. |
| h | int | **타깃 변수(bbox)**. 박스 높이(px)입니다. |
| area | int | 박스 넓이(px²). 대부분 `w × h`와 같습니다. 다른 경우는 5.4에서 봅니다. |
| iscrowd, ignore, has_segmentation | int, int, bool | 평가 분류, 무시 여부, segmentation 유무입니다. |

### 약 식별 코드

| 컬럼명 | 데이터 타입 | 설명 |
|---|---|---|
| drug_N | str | 약 코드(예: `K-003351`). `category_id`에 `K-`를 붙인 형태입니다. |
| dl_mapping_code | str | 약 매핑 코드. `drug_N`과 같은 값입니다. |
| dl_idx | str | 약 코드의 숫자 부분(예: `3351`)입니다. (이 부분은 일단 눈여겨 보자) |
| item_seq | int | 식약처 품목기준코드입니다. |
| di_edi_code | str | 건강보험 EDI 코드. 여러 개면 쉼표로 구분합니다. |

### 약 기본 정보

| 컬럼명 | 데이터 타입 | 설명 |
|---|---|---|
| dl_name | str | 약 이름(한글)입니다. |
| dl_name_en | str | 약 이름(영문). 14행이 빈 문자열입니다. |
| dl_material | str | 주성분(한글). 성분이 여러 개면 `\|`로 구분합니다. 9행이 빈 문자열입니다. |
| dl_material_en | str | 주성분(영문)입니다. |
| dl_company | str | 제약회사(한글)입니다. |
| dl_company_en | str | 제약회사(영문)입니다. |
| di_company_mf | str | 위탁 제조사(한글). 대부분(637행) 빈 문자열입니다. |
| di_company_mf_en | str | 위탁 제조사(영문). 대부분(724행) 빈 문자열입니다. |
| di_class_no | str | 약효 분류(예: `[02180]동맥경화용제`)입니다. |
| di_etc_otc_code | str | 의약품 구분. `전문의약품` 또는 `일반의약품`입니다. |
| di_item_permit_date | str | 품목 허가일(YYYYMMDD)입니다. |

### 약 외형

| 컬럼명 | 데이터 타입 | 설명 |
|---|---|---|
| dl_custom_shape | str | 제형 분류(예: `정제, 저작정`, `경질캡슐제`)입니다. |
| form_code_name | str | 제형 이름(예: `필름코팅정`, `나정`)입니다. |
| drug_shape | str | 알약 모양(예: `원형`, `타원형`, `장방형`)입니다. |
| chart | str | 성상. 알약 생김새를 글로 설명한 것입니다. |
| color_class1 | str | 주 색상입니다. |
| color_class2 | str | 보조 색상. 대부분 빈 문자열입니다. |
| thick | float | 두께(mm). 0.0은 값이 없는 경우입니다. |
| leng_long | float | 장축 길이(mm). 0.0은 값이 없는 경우입니다. |
| leng_short | float | 단축 길이(mm). 0.0은 값이 없는 경우입니다. |
| print_front | str | 앞면 각인 문자(예: `R5`)입니다. |
| print_back | str | 뒷면 각인 문자. `분할선`처럼 적힌 경우도 있습니다. |
| line_front | str | 앞면 분할선. `-`(일자), `+`(십자), 빈 문자열(없음)입니다. |
| line_back | str | 뒷면 분할선. 값은 `line_front`와 같은 방식입니다. |

### 마크 정보

| 컬럼명 | 데이터 타입 | 설명 |
|---|---|---|
| mark_code_front_anal | str | 앞면 마크 설명(예: `pfizer`). 대부분 빈 문자열입니다. |
| mark_code_front_img | str | 앞면 마크 이미지 URL. 대부분 빈 문자열입니다. |
| mark_code_back_anal | str | 뒷면 마크 설명. 전부 비어 있습니다. |
| mark_code_back_img | str | 뒷면 마크 이미지 URL. 전부 비어 있습니다. |
| mark_code_front | str | 앞면 마크 코드. 전부 비어 있습니다. |
| mark_code_back | str | 뒷면 마크 코드. 전부 비어 있습니다. |

### 기타

| 컬럼명 | 데이터 타입 | 설명 |
|---|---|---|
| img_key | str | 공공 낱알 이미지 URL입니다. |
| img_regist_ts | str | 낱알 이미지 등록일(YYYYMMDD)입니다. |
| change_date | str | 약 정보 변경일(YYYYMMDD)입니다. |

> 참고: `img_key`, `chart`, `drug_shape`, `print_*`, `color_*`, `line_*`, `mark_*`, `img_regist_ts`,
> `change_date`, `form_code_name`은 18행이 결측(NaN)입니다. 키는 있고 값이 `null`입니다(4.3에서 확인).
> 원본 kwon 정의서와 dongwook 셀 9는 이를 "키 없음"으로 적었습니다.

## 3. 데이터 규모와 구조

> 이식 예정입니다. 아래 항목과 기준본 셀(원본 노트북 셀 번호)을 옮깁니다.

- 3.1 파일 수, 해상도, 색상 모드, 디코딩 오류 — JJAM 5 기준, sohee_eda 11의 디코딩 실패·SHA256
- 3.2 1 사진 : N JSON 구조와 bbox 시각 확인 — JJAM 7, 10, 13
- 3.3 JSON 내용·라벨·픽셀 중복, train/test 중복 — JJAM 16, 19, sohee_eda 11의 train/test 공통 SHA256

## 4. 결측

> 이식 예정입니다. 아래 항목과 기준본 셀(원본 노트북 셀 번호)을 옮깁니다.

- 4.1 bbox 값 결측 — dongwook 6
- 4.2 라벨 누락 (파일명에 있는 약의 bbox 없음) — kwon 5–8
- 4.3 메타데이터 결측 — dongwook 9 기준, JJAM 35·37(null 시각화), eda_sohee 10(결측 이유 표)

## 5. 라벨 오류와 이상치

> 이식 예정입니다. 아래 항목과 기준본 셀(원본 노트북 셀 번호)을 옮깁니다.

- 5.1 이미지 밖 좌표 — JJAM 22, 28, 31
- 5.2 다른 약과 좌표 중복 — kwon 58–59, jang 9의 정답 약 판정
- 5.3 빈 배경 박스 — kwon 70–73
- 5.4 area ≠ w×h — JJAM 46
- 5.5 면적·가로세로 비율 이상치 후보 (IQR, 오류 아님) — JJAM 22, eda_sohee 19–22
- 5.6 붙은·부서진 알약 — kwon 54–66
- 5.7 bbox 여백 (GrabCut 추정) [RUN_HEAVY] — JJAM 54–66
- 5.8 라벨 오류 요약표 — 문제 유형, 건수, 처리 방식, 보정 파일

## 6. 분포

> 이식 예정입니다. 아래 항목과 기준본 셀(원본 노트북 셀 번호)을 옮깁니다.

- 6.1 클래스별 bbox 수와 불균형 단계 — kwon 35, 37, jang 15
- 6.2 bbox 크기 분포, 클래스별 크기 — jang 13, 17

## 7. 촬영 조건

> 이식 예정입니다. 아래 항목과 기준본 셀(원본 노트북 셀 번호)을 옮깁니다.

- 7.1 조합별 각도 구성 — kwon 41, 43
- 7.2 각도·배경·조명별 bbox 크기와 비율 — dongwook 11, jang 19–20
- 7.3 알약 실제 크기와 픽셀 크기 — dongwook 13–17 (JJAM 과거 분석은 Git 이력에서 확인 후 보완)

## 8. 데이터 누수와 분할

> 이식 예정입니다. 아래 항목과 기준본 셀(원본 노트북 셀 번호)을 옮깁니다.

- 8.1 희귀 약은 사실상 장면 하나 — kwon 46–53
- 8.2 같은 조합 유사도 (pHash) — kwon 90–107, sohee_eda 15의 aHash와 비교
- 8.3 다른 조합의 세팅 공유와 묶음 — kwon 109–118
- 8.4 분할 후보 비교 — kwon 120, sohee_eda 17, jang 20

## 9. 결론과 다음 작업

> 이식 예정입니다. 아래 항목과 기준본 셀(원본 노트북 셀 번호)을 옮깁니다.

- 확정된 사실
- 결정할 것
- 학습 파이프라인으로 넘길 것